# 01_cam_qualitative_cls_gap_mel_nv

Generate qualitative CAM panels and a PDF for the clean MEL vs NV PanDerm models.

Rows per image:
- CLS checkpoint
- GAP checkpoint

Columns per row:
- RGB image with lesion outline
- Grad CAM target class
- Grad CAM reference class
- Difference CAM
- Finer CAM

Main options:
- `SAMPLE_MODE = "qualitative"` uses 10 MEL + 10 NV from the qualitative CSV.
- `SAMPLE_MODE = "all_test"` creates CAMs for all test images.
- `TARGET_BLOCK_INDICES` can contain one or multiple block indices, e.g. `[-1]`, `[-6, -1]`, or `[-10, -1]`.


In [ ]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS
# =============================================================================

# Notebook location: notebooks/mel_nv/01_cam_qualitative_cls_gap_mel_nv.ipynb
# REPO_ROOT = Path("../..").resolve() # local notebook
REPO_ROOT = Path("..").resolve() # ubelix notebook
REPO_ROOT = REPO_ROOT / "master-thesis"

HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# Choose:
# - "qualitative": uses 10 MEL + 10 NV selected in notebook 00
# - "all_test": uses every test image from ham_mel_nv_clean.csv
SAMPLE_MODE = "qualitative"  # "qualitative" or "all_test"

# Use one or multiple blocks.
# Recommended first run: [-1]
# Optional internal comparison: CLS [-6, -1], GAP [-10, -1]
TARGET_BLOCK_INDICES = [-1]

# Set True first if you only want to inspect commands.
DRY_RUN = False

# If SAMPLE_MODE="qualitative", this limits the qualitative CSV.
# If None, use all rows in the selected CSV.
NUM_SAMPLES_OVERRIDE = None

# Panel columns. Keep this 5-column version for your thesis/internal review.
PANEL_ITEMS = "rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

# Finer-CAM comparison strength.
ALPHA = 0.8

# Paths
CLEAN_CSV = MEL_NV_ROOT / "ham_mel_nv_clean.csv"
QUAL_CSV = MEL_NV_ROOT / "ham_mel_nv_clean_qualitative_10_per_class_seed42.csv"

CLS_CKPT = REPO_ROOT / "external" / "checkpoints5" / "checkpoint-best-cls-ha5.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints5" / "checkpoint-best-gap-ha5.pth"

GT_COL = "gt_label"
CLASS_ARGS = ["--class_names", "MEL,NV"]
COMPARE_ARGS = [
    "--compare_mode", "gt_pair",
    "--A", "MEL",
    "--B", "NV",
    "--topk_compare", "1",
]

SCENARIOS = [
    {
        "name": "CLS HA 5.0",
        "checkpoint": CLS_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "cls",
    },
    {
        "name": "GAP HA 5.0",
        "checkpoint": GAP_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "mean",
    },
]

print("REPO_ROOT:", REPO_ROOT)
print("CLEAN_CSV exists:", CLEAN_CSV.exists(), CLEAN_CSV)
print("QUAL_CSV exists:", QUAL_CSV.exists(), QUAL_CSV)
print("CLS_CKPT exists:", CLS_CKPT.exists(), CLS_CKPT)
print("GAP_CKPT exists:", GAP_CKPT.exists(), GAP_CKPT)
for scenario in SCENARIOS:
    print("\n", scenario["name"])
    print("  checkpoint:", scenario["checkpoint"])
    print("  pooling:", scenario["pooling"])
    if not Path(scenario["checkpoint"]).exists():
        print("  [WARN] missing checkpoint")


REPO_ROOT: /storage/homefs/cn21m021/projects/master-thesis
CLEAN_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean.csv
QUAL_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
CLS_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth
GAP_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-gap-ha5.pth

 CLS HA 5.0
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth
  pooling: cls

 GAP HA 5.0
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-gap-ha5.pth
  pooling: mean


## 2. Build the active CSV

For `all_test`, this cell creates a temporary CSV containing only test images. This avoids accidentally processing train/val rows.

In [2]:
def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "."]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def make_active_csv_for_block(block_index: int) -> tuple[Path, int, Path]:
    out_root = REPO_ROOT / "outputs" / "mel_nv" / f"cam_qualitative_cls_gap_{SAMPLE_MODE}_block{block_index}"
    out_root.mkdir(parents=True, exist_ok=True)
    csv_out_dir = out_root / "csv"
    csv_out_dir.mkdir(parents=True, exist_ok=True)

    if SAMPLE_MODE == "qualitative":
        active_csv = QUAL_CSV
        df = pd.read_csv(active_csv)
    elif SAMPLE_MODE == "all_test":
        df = pd.read_csv(CLEAN_CSV)
        df = df[df["split"].astype(str).str.lower().eq("test")].copy()
        df = df.sort_values(["gt_label", "image_id"]).reset_index(drop=True)
        active_csv = csv_out_dir / "ham_mel_nv_clean_all_test.csv"
        df.to_csv(active_csv, index=False)
    else:
        raise ValueError("SAMPLE_MODE must be 'qualitative' or 'all_test'.")

    if NUM_SAMPLES_OVERRIDE is None:
        num_samples = len(df)
    else:
        num_samples = min(int(NUM_SAMPLES_OVERRIDE), len(df))

    print("\nBlock:", block_index)
    print("OUT_ROOT:", out_root)
    print("ACTIVE_CSV:", active_csv)
    print("NUM_SAMPLES:", num_samples)
    display(df.head())
    display(df.head(num_samples).groupby(["split", "gt_label"]).size().unstack(fill_value=0))

    return active_csv, num_samples, out_root


## 3. Generate CAM panels

This calls `scripts.generate_finer_cam_panderm` once per scenario and block.

Important: this notebook assumes `scripts.generate_finer_cam_panderm.py` supports `--pooling`.

In [3]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_panels_for_block(block_index: int, active_csv: Path, num_samples: int, out_root: Path):
    panel_root = out_root / "panels"
    panel_root.mkdir(parents=True, exist_ok=True)

    for scenario in SCENARIOS:
        scenario_name = scenario["name"]
        scenario_out_dir = panel_root / safe_name(scenario_name)
        scenario_out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(active_csv),
            "--image_col", "image_rel_path",
            "--img_dir", str(IMG_DIR),
            "--gt_col", GT_COL,
            "--checkpoint", str(scenario["checkpoint"]),
            "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
            "--pooling", scenario["pooling"],
            "--out_dir", str(scenario_out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--alpha", str(ALPHA),
            "--panel_items", PANEL_ITEMS,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--target_block_index", str(block_index),
            "--clinician_labels",
            "--model_display_name", scenario_name,
            # "--save_json",
            "--save_raw_cams",
        ]

        cmd += CLASS_ARGS
        cmd += COMPARE_ARGS

        print(f"\nGenerating panels: {scenario_name} | block {block_index}")
        run_command(cmd, dry_run=DRY_RUN)


BLOCK_RUNS = []
for block_index in TARGET_BLOCK_INDICES:
    active_csv, num_samples, out_root = make_active_csv_for_block(block_index)
    generate_panels_for_block(block_index, active_csv, num_samples, out_root)
    BLOCK_RUNS.append({"block_index": block_index, "active_csv": active_csv, "num_samples": num_samples, "out_root": out_root})



Block: -1
OUT_ROOT: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1
ACTIVE_CSV: /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
NUM_SAMPLES: 20


,lesion_id,image_id,image,dx,gt_label,label,label_2class,binary_label,split,image_rel_path,...,cue_applied,cue_mask_rel_path,cue_mode,dx_type,age,sex,localization,dataset,age_group,dx_norm
0,HAM_0005846,ISIC_0024459,ISIC_0024459.jpg,mel,MEL,4,0,0,test,images/ISIC_0024459.jpg,...,False,NaN,clean,histo,80.0,male,back,vienna_dias,old,mel
1,HAM_0007272,ISIC_0024756,ISIC_0024756.jpg,mel,MEL,4,0,0,test,images/ISIC_0024756.jpg,...,False,NaN,clean,histo,60.0,male,lower extremity,rosendahl,old,mel
2,HAM_0002576,ISIC_0025414,ISIC_0025414.jpg,mel,MEL,4,0,0,test,images/ISIC_0025414.jpg,...,False,NaN,clean,histo,55.0,male,lower extremity,rosendahl,old,mel
3,HAM_0007031,ISIC_0025616,ISIC_0025616.jpg,mel,MEL,4,0,0,test,images/ISIC_0025616.jpg,...,False,NaN,clean,histo,70.0,female,upper extremity,rosendahl,old,mel
4,HAM_0006696,ISIC_0026094,ISIC_0026094.jpg,mel,MEL,4,0,0,test,images/ISIC_0026094.jpg,...,False,NaN,clean,histo,20.0,male,back,rosendahl,young,mel


gt_label,MEL,NV
split,,
test,10,10



Generating panels: CLS HA 5.0 | block -1

python -m scripts.generate_finer_cam_panderm --csv /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv --image_col image_rel_path --img_dir /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth --checkpoint_model_type panderm --pooling cls --out_dir /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/cls_ha_5_0 --num_samples 20 --method finercam --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam --mask_root /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -1 --clinician_labels --model_display_name 'CLS HA 5.0' --save_raw_cams --class_names MEL,NV --compare_mode gt_pair --A MEL --B NV --topk_compar

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[warn] load_state_dict mismatch: missing=2, unexpected=2
  missing sample: ['norm.weight', 'norm.bias']
  unexpected sample: ['fc_norm.weight', 'fc_norm.bias']
[info] Loaded PanDerm Base FT from checkpoint-best-cls-ha5.pth
[info] checkpoint_model_type= panderm pooling= cls use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[11].norm1 (requested -1)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/cls_ha_5_0/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.991, NV: 0.009]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/cls_ha_5_0/raw_cams/ISIC_0

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[info] Loaded PanDerm Base FT from checkpoint-best-gap-ha5.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[11].norm1 (requested -1)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/gap_ha_5_0/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.894, NV: 0.106]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/gap_ha_5_0/raw_cams/ISIC_0024756
[info] images/ISIC_0024756.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.564, NV: 0.436]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 

## 4. Build PDF

One PDF page per image. Each page has one row per model.

In [4]:
from PIL import Image, ImageDraw, ImageFont


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(24, bold=True)
FONT_SMALL = get_font(18, bold=False)


def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def load_display_rows_for_pdf(active_csv: Path, num_samples: int) -> pd.DataFrame:
    df = pd.read_csv(active_csv).head(num_samples).copy()
    if "image_id" not in df.columns:
        if "image_rel_path" in df.columns:
            df["image_id"] = df["image_rel_path"].apply(lambda x: Path(str(x)).stem)
        elif "image" in df.columns:
            df["image_id"] = df["image"].apply(lambda x: Path(str(x)).stem)
        else:
            raise ValueError("Need one of image_id, image_rel_path, or image columns.")
    return df


def find_panel_png(panel_root: Path, scenario_name: str, row: pd.Series) -> Path | None:
    out_dir = panel_root / safe_name(scenario_name)
    candidates = []

    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct
        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]

    return None


def wrap_text(draw, text: str, font, max_width: int) -> list[str]:
    words = str(text).split()
    lines = []
    current = ""
    for word in words:
        test = f"{current} {word}".strip()
        bbox = draw.textbbox((0, 0), test, font=font)
        if bbox[2] - bbox[0] <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


def make_page_for_image(row: pd.Series, panel_root: Path, block_index: int) -> Image.Image:
    page_width = 2400
    margin = 50
    label_width = 300
    gap = 18
    title_h = 120
    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_panels = []
    for scenario in SCENARIOS:
        panel_path = find_panel_png(panel_root, scenario["name"], row)
        if panel_path is None:
            loaded_panels.append((scenario, None, None))
            continue
        panel = Image.open(panel_path).convert("RGB")
        scale = available_panel_width / panel.width
        new_h = int(panel.height * scale)
        panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)
        loaded_panels.append((scenario, panel, panel_path))

    row_heights = [panel.height if panel is not None else 260 for _, panel, _ in loaded_panels]
    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin
    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    title = f"MEL vs NV CAM review: {image_id}"
    subtitle = f"Ground truth: {gt} | Target block: {block_index} | Mode: {SAMPLE_MODE}"
    draw.text((margin, 30), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 78), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h
    for idx, (scenario, panel, panel_path) in enumerate(loaded_panels, start=1):
        row_h = row_heights[idx - 1]
        label_lines = wrap_text(draw, scenario["name"], FONT_LABEL, label_width - 10)
        label_x = margin
        label_y = y + 25
        for line in label_lines:
            draw.text((label_x, label_y), line, fill="black", font=FONT_LABEL)
            label_y += 32
        draw.text((label_x, label_y + 10), f"Block {block_index}", fill=(80, 80, 80), font=FONT_SMALL)
        draw.text((label_x, label_y + 36), f"Pooling: {scenario['pooling']}", fill=(80, 80, 80), font=FONT_SMALL)

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle([box_x, box_y, box_x + available_panel_width, box_y + row_h], outline=(180, 180, 180), width=2)
            draw.text((box_x + 30, box_y + 80), "Missing panel PNG", fill=(160, 0, 0), font=FONT_LABEL)
        else:
            page.paste(panel, (margin + label_width + gap, y))
        y += row_h + gap

    return page


def build_pdf_for_block(block_run: dict):
    block_index = block_run["block_index"]
    active_csv = block_run["active_csv"]
    num_samples = block_run["num_samples"]
    out_root = block_run["out_root"]
    panel_root = out_root / "panels"

    pdf_out = out_root / f"mel_nv_cam_cls_gap_{SAMPLE_MODE}_block{block_index}.pdf"
    config_out = out_root / f"mel_nv_cam_cls_gap_config_{SAMPLE_MODE}_block{block_index}.json"

    rows = load_display_rows_for_pdf(active_csv, num_samples)
    pages = [make_page_for_image(row, panel_root, block_index) for _, row in rows.iterrows()]
    if not pages:
        raise RuntimeError("No pages generated.")

    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)

    config = {
        "sample_mode": SAMPLE_MODE,
        "target_block_index": block_index,
        "num_samples": num_samples,
        "active_csv": str(active_csv),
        "pdf_out": str(pdf_out),
        "out_root": str(out_root),
        "panel_items": PANEL_ITEMS,
        "alpha": ALPHA,
        "scenarios": [
            {
                "name": s["name"],
                "checkpoint": str(s["checkpoint"]),
                "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
                "pooling": s["pooling"],
            }
            for s in SCENARIOS
        ],
        "class_args": CLASS_ARGS,
        "compare_args": COMPARE_ARGS,
    }
    config_out.write_text(json.dumps(config, indent=2))
    print("Saved PDF:", pdf_out)
    print("Saved config:", config_out)


for block_run in BLOCK_RUNS:
    if DRY_RUN:
        print("DRY_RUN=True, skipping PDF build for block", block_run["block_index"])
    else:
        build_pdf_for_block(block_run)


Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/mel_nv_cam_cls_gap_qualitative_block-1.pdf
Saved config: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/mel_nv_cam_cls_gap_config_qualitative_block-1.json


## 5. Quick checks

Run this if you want to inspect panel counts per scenario.

In [5]:
for block_run in BLOCK_RUNS:
    block_index = block_run["block_index"]
    out_root = block_run["out_root"]
    panel_root = out_root / "panels"
    print("\n" + "=" * 80)
    print("Block:", block_index)
    print("OUT_ROOT:", out_root)
    for scenario in SCENARIOS:
        out_dir = panel_root / safe_name(scenario["name"])
        pngs = sorted(out_dir.glob("*.png"))
        metas = sorted(out_dir.glob("*_meta.json"))
        raw_dir = out_dir / "raw_cams"
        raw_files = sorted(raw_dir.glob("*.npy")) if raw_dir.exists() else []
        print("\n" + scenario["name"])
        print("  out_dir:", out_dir)
        print("  png panels:", len(pngs))
        print("  meta files:", len(metas))
        print("  flat raw cam files:", len(raw_files))
        for p in pngs[:3]:
            print("   ", p.name)



Block: -1
OUT_ROOT: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1

CLS HA 5.0
  out_dir: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/cls_ha_5_0
  png panels: 20
  meta files: 0
  flat raw cam files: 80
    ISIC_0024459_rgb_gt_mask_gradcam_a_gradcam_b_map_diff_finercam.png
    ISIC_0024756_rgb_gt_mask_gradcam_a_gradcam_b_map_diff_finercam.png
    ISIC_0025414_rgb_gt_mask_gradcam_a_gradcam_b_map_diff_finercam.png

GAP HA 5.0
  out_dir: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/cam_qualitative_cls_gap_qualitative_block-1/panels/gap_ha_5_0
  png panels: 20
  meta files: 0
  flat raw cam files: 80
    ISIC_0024459_rgb_gt_mask_gradcam_a_gradcam_b_map_diff_finercam.png
    ISIC_0024756_rgb_gt_mask_gradcam_a_gradcam_b_map_diff_finercam.png
    ISIC_0025414_rgb_gt_mask_gradcam_a_gradcam_b_map_diff_finercam.png
